# 第 4 章实践：GEMM

本节请独立完成三个版本的矩阵乘：单 Tile、K 分块累加、多核动态 M/N。每题都明确输入输出、启动方式和验收条件，并提供可直接调用的 Host 校验器。无需运行前面几个 Notebook；完成本页环境准备即可开始。

建议先写出地址预算和循环边界，再补 Kernel。全部答案放在末尾；在运行环境不可用时，可先检查推导与源码，但应把设备验收状态保留为“待验证”。


## 环境准备

运行代码前，请完成 [01.06 环境配置](../01_pypto_pro_introduction/01.06_env_and_first_kernel.ipynb)。

在本节所在目录启动 Notebook，按顺序执行单元即可；不需要先运行其他 Notebook。设备号默认取 `TILE_FWK_DEVICE_ID`，未设置时使用 0。


In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

device = f"npu:{int(os.environ['TILE_FWK_DEVICE_ID'])}"
torch.npu.set_device(device)


## 1. 客观题

1. （填空）矩阵乘默认进入 Cube 的分形路径：A 用 ______，B 用 ______，结果 C 用 ______。
2. （选择）K 分块首块使用 `matmul`、后续使用 `matmul_acc`，是因为：A. 后者总是更快；B. 前者初始化结果，后者累加已有结果；C. 前者不能处理大矩阵。
3. （判断）L1 的 `0x00000` 与 L0A 的 `0x0000` 是同一段内存。（对 / 错）
4. （填空）100 个任务、32 核按连续区间分配，每核任务上限为 ______；实际执行任务的核数为 ______。


In [ ]:
!cat ./answer/04.05_objective_answer.txt


## 2. 共用校验器：先约定怎样算完成

三个 Kernel 的调用形式统一为 `kernel[None, cores](a, b, out)`，输出由调用者预先分配，Kernel 原地写入，不需要返回 Tensor。输入是连续的 FP16，输出是 FP32。

校验器会做四件事：构造输入；用 NaN 初始化输出；启动后同步；对照 FP32 Golden 检查全量结果。每个用例有明确目的：
<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">输入 / 形状</th><th style="text-align:left;">重点检查</th></tr>
<tr><td>随机值，固定种子</td><td>一般数值正确性；检测行列错位与错误的读取范围</td></tr>
<tr><td>全 1</td><td>单 Tile 的输出是否恰好等于 K</td></tr>
<tr><td>K 三块分别为 1、2、3</td><td>是否正确累加全部 K 块，结果应为 768</td></tr>
<tr><td>128×128，4 核</td><td>任务少于核数时，空闲核是否安全退出</td></tr>
<tr><td>512×300 与 500×300</td><td>N 尾块以及 M/N 同时存在尾块</td></tr>
<tr><td>同一形状使用 1、4、8 核</td><td>切分变化后是否仍覆盖全部输出</td></tr>
</table>


随机输入沿用 `rtol=1e-2, atol=1e-2`；全 1 和分块常量的期望值可精确表示，再增加严格等值检查。未填写 Kernel 时，验收单元会提示“尚未完成”，不会打印 PASS。


In [ ]:
def check_case(kernel, m, k, n, cores=1, *, pattern="random", transpose_b=False):
    """在 950 上执行一组用例；NaN 初值帮助发现输出漏写。"""
    if min(m, k, n, cores) <= 0:
        raise ValueError("Shapes and core count must be positive")
    device = f"npu:{int(os.environ['TILE_FWK_DEVICE_ID'])}"
    torch.npu.set_device(device)
    torch.manual_seed(42)
    if pattern == "random":
        a = torch.randn(m, k, device=device, dtype=torch.float16)
        b = torch.randn(k, n, device=device, dtype=torch.float16)
    elif pattern == "k_blocks":
        if k != 384:
            raise ValueError("k_blocks requires K=384")
        a = torch.ones(m, k, device=device, dtype=torch.float16)
        b = torch.ones(k, n, device=device, dtype=torch.float16)
        b[128:256, :] = 2
        b[256:384, :] = 3
    elif pattern == "ones":
        a = torch.ones(m, k, device=device, dtype=torch.float16)
        b = torch.ones(k, n, device=device, dtype=torch.float16)
    else:
        raise ValueError(f"Unknown pattern: {pattern}")
    out = torch.full((m, n), float("nan"), device=device, dtype=torch.float32)
    rhs = b.t().contiguous() if transpose_b else b
    kernel[None, cores](a, rhs, out)
    torch.npu.synchronize()
    assert out.shape == (m, n) and out.dtype == torch.float32
    assert torch.isfinite(out).all().item(), "Output contains NaN/Inf or unwritten elements"
    golden = a.float() @ b.float()
    torch.testing.assert_close(out, golden, rtol=1e-2, atol=1e-2)
    if pattern in {"ones", "k_blocks"}:
        expected = 768.0 if pattern == "k_blocks" else float(k)
        torch.testing.assert_close(out, torch.full_like(out, expected), rtol=0, atol=0)
    max_abs = (out - golden).abs().max().item()
    print(f"PASS M={m}, K={k}, N={n}, cores={cores}, {pattern}, max_abs={max_abs:.6g}")


## 3. 实践一：单核单 Tile（简单）

**规格**：`A[128,128] × B[128,128] → C[128,128]`，FP16 输入、FP32 输出，只用 1 核。

**要完成的工作**：创建 Mat×2、Left、Right、Acc 共 5 个单槽 TileGroup；使用自动同步；完成两次 load、两次 move、一次 matmul 和一次 store。

**动手前先写出**：每个输入 Tile 占多少字节？A、B 同处 L1 时，B 最早能放在哪里？L0A 与 L0B 能否同时从零开始？

在下方工作区实现名为 `student_single` 的 Kernel。将 `None` 占位替换为完整函数定义，类型签名可按注释填写；随后运行本题验收单元。


In [ ]:
# 用你的完整 Kernel 定义替换下面的占位。
student_single = None

# @pl.jit(auto_mutex=True)
# def student_single(
#     a: pl.Tensor[[128, 128], pl.DT_FP16],
#     b: pl.Tensor[[128, 128], pl.DT_FP16],
#     out: pl.Tensor[[128, 128], pl.DT_FP32],
# ):
#     创建 TileGroup，并在 section_cube 中完成数据路径。


In [ ]:
if student_single is None:
    print('实践一尚未完成：请填写 student_single；当前未做设备验收')
else:
    check_case(student_single, 128, 128, 128)
    check_case(student_single, 128, 128, 128, pattern='ones')


**验收条件**：随机输入通过容差检查；全 1 输入的每个输出元素均为 128；输出没有 NaN/Inf。若失败，先检查 MemorySpace、L1 地址重叠和搬运顺序，再看算术。


## 4. 实践二：K 分块累加（中等）

**规格**：`A[128,384] × B[384,128] → C[128,128]`，用 1 核，K 每次处理 128。

**要完成的工作**：循环完成 3 个部分乘积；首块初始化，后续块累加；循环结束后只写回一次。L1 与 L0A/L0B 的输入组使用双缓冲，L0C 保持一个累加器。

**动手前先写出**：A 的两个 L1 槽位占用什么区间？B 的起点为什么不能沿用实践一？累加器的 `current()` 应放在 K 循环内还是外？在 04.02 的示例中，它放在循环外以明确同一结果的生命周期。


In [ ]:
student_k_split = None

# @pl.jit(auto_mutex=True)
# def student_k_split(
#     a: pl.Tensor[[128, 384], pl.DT_FP16],
#     b: pl.Tensor[[384, 128], pl.DT_FP16],
#     out: pl.Tensor[[128, 128], pl.DT_FP32],
# ):
#     规划双缓冲；按 k=0,128,256 搬入；首块初始化、后续累加。


In [ ]:
if student_k_split is None:
    print('实践二尚未完成：请填写 student_k_split；当前未做设备验收')
else:
    check_case(student_k_split, 128, 384, 128)
    check_case(student_k_split, 128, 384, 128, pattern='k_blocks')


**验收条件**：随机输入通过容差检查；三个 K 块分别贡献 128、256、384，最终每个元素均为 768。请在实验记录中解释：如果得到 384，优先检查什么？双缓冲是否已经证明更快？后一个问题需要性能实测，数值通过本身不能回答。


## 5. 实践三：多核动态 M/N 与尾块（困难）

**基础规格**：`M=512,N=300,K=128`，FP16 输入、FP32 输出，4 核启动。进一步要求同一个 Kernel 支持本页验收用例中的其他正整数 M/N，K 保持 128。

**要完成的工作**：

1. 以 `pl.DYNAMIC` 声明 M/N，从 Tensor shape 获取实际尺寸。
2. 使用 128×128 输出 Tile，向上取整计算块数，用任务编号和连续区间分配工作。
3. 对 5 个 Tile 在每轮搬运/计算前设置一致的有效形状，处理完整块与尾块之间的切换。
4. 保证各核输出互不重叠，任务少于核数时空闲核不访问数据。

**动手前先写出**：基础规格对应 4×3=12 个任务，4 核各领哪些任务？把 M 改成 500 后，任务总数是否改变？右下角的有效形状变成什么？


In [ ]:
student_dynamic = None

# @pl.jit(auto_mutex=True)
# def student_dynamic(
#     a: pl.Tensor[[pl.DYNAMIC, 128], pl.DT_FP16],
#     b: pl.Tensor[[128, pl.DYNAMIC], pl.DT_FP16],
#     out: pl.Tensor[[pl.DYNAMIC, pl.DYNAMIC], pl.DT_FP32],
# ):
#     计算核区间；规划 Tile；逐任务设置有效范围并完成矩阵乘。


In [ ]:
if student_dynamic is None:
    print('实践三尚未完成：请填写 student_dynamic；当前未做设备验收')
else:
    for m, n, cores in ((128, 128, 4), (256, 256, 4), (512, 300, 4),
                        (500, 300, 1), (500, 300, 4), (500, 300, 8)):
        check_case(student_dynamic, m, 128, n, cores)


**验收条件**：所有形状与启动核数组合均通过全量结果校验。记录设备与软件版本、用例、实际误差和是否通过；尚未执行的项保留“待验证”。若只通过基础规格，不能据此认定双方向尾块或空闲核场景也正确。

### 参考答案

完成尝试后再查看下面的设计说明和 [完整参考源码](./src/gemm_reference.py)。源码包含本章的单 Tile、K 分块、转置 B、动态多核四个 Kernel，以及与本页一致的校验器；可在 950 环境从章节目录运行：

```bash
python src/gemm_reference.py
```

该脚本不依赖其他 Notebook 或本地 PyPTO 源码仓路径。以下 `cat` 单元只展示内容，不会执行设备计算。


In [ ]:
!cat ./answer/04.05_practice_answer.txt


In [ ]:
!cat ./src/gemm_reference.py


## 6. 本章回顾

完成三题后，应能从一个矩阵规格推导出 Tile 容量、片上地址、K 累加顺序、核间任务区间和尾块有效形状，并为这些设计准备对应的校验用例。

下一章将把 Cube 的矩阵结果交给 Vector 继续处理。届时除了本章的核内流水依赖，还要说明不同执行域之间的数据交接与同步。
